# Experiment results: data sparsity and label noise

One figure per experiment, one panel per AntMaze variant, one series per reward
model. Each point is the **mean over seeds 1-10** of the last-10 evaluation score
(`results/iql_score.py`, × 100), with the **seed-to-seed standard deviation** as
the error bar, the convention Chapter 3 registers for reporting. Lines connect a
model's points across levels.

**The x-axis runs from full data, on the left, to the most degraded data, on the
right**:

- **Sparsity** (Experiment 1): full train-pool, then 128, 64, 32 and 16
  preference pairs. Equal steps are halvings, except the first, whose size
  depends on the variant's pool (254-514 pairs; shown on the tick).
- **Label noise** (Experiment 2, primary arm): flip fraction 0 (the anchor), 0.1,
  0.2, 0.3, 0.4, 0.45, placed at their values.

The full-data point of every series is the anchor evaluation (`results_table.ipynb`).
The task reward has no condition of its own, so its anchor result is drawn as a
dashed reference line across each panel.

**This figure is descriptive, not a test.** The registered reading is
`python results/exp_readout.py --axis sparsity` (or `--axis noise`),
handoff §13.9 and §13.11. The figure applies the same gates:

- **Only complete, config-audited cells are drawn**, meaning ten seeds × 200
  evaluation points, with every run's config matching the generated sweep file
  (§12.5). Each cell is found by an exact parameter match, through the same code
  the readout uses, so nothing is registered by hand.
- **H3 calibration gate** (§13.10, §13.12): a BNN point whose (variant, level)
  gate is not PASS is drawn **hollow**. It is conditional, flagged and not
  dropped. A BNN point whose gate has not been read yet is **not drawn**, because
  the gate is read before the BNN's policy results.

The last two sections draw the same figures **without terminals** (§14.5's
candidate arm), each reward at its own no-terminals index.
The final section draws the **reward models' own accuracy** on the clean validation
split along both ladders (RQ1's manipulation check, §14.5).

Run from `results/` with the `irl` environment. Fetching takes a few minutes.
Commit with all outputs empty.

In [ ]:
import math
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb

sys.path.insert(0, os.path.abspath(".."))  # phase2_sweeps
import eval_readout as R  # noqa: E402  the registered machinery (§12.5 gates)
import exp_readout as X  # noqa: E402

_argv, sys.argv = sys.argv, [sys.argv[0]]
import phase2_sweeps as P  # noqa: E402

sys.argv = _argv

SCALE = 100.0  # success rate -> 0-100 score
WINNERS = __import__("json").load(open(os.path.join("..", "phase2_winners.json")))
# the no-terminals grid winners: the candidate protocol's frozen indices (§14.5)
GRID_WINNERS = __import__("json").load(open(os.path.join("..", P.GRID_WINNERS)))
api = wandb.Api(timeout=120)

## Configuration

`METHODS` fixes the series order, labels and colours. They match
`results_table.ipynb` (same order, same colour cycle), so a model looks the same
in both notebooks.

In [ ]:
METHODS = [
    ("task_reward", "Task reward (full data)"),
    ("MR", "MR"),
    ("PT", "PT"),
    ("NMR", "NMR"),
    ("ENS_MEAN", "ENS w/ MEAN"),
    ("ENS_CVAR", "ENS w/ CVaR"),
    ("BNN_MEAN", "BNN w/ MEAN"),
    ("BNN_CVAR", "BNN w/ CVaR"),
]
BNN = ("BNN_MEAN", "BNN_CVAR")  # the series the H3 gate applies to
# The BNN/CVaR hold (gp_reward-priors HANDOFF §14.1): covered series are dropped, and
# the readout collectors do not fetch them (results/hold.py).  Set INCLUDE_HELD = True
# only once the author has lifted the hold.
INCLUDE_HELD = False
HELD = ("BNN_CVAR", "BNN_MEAN", "ENS_CVAR")
if INCLUDE_HELD:
    import hold

    hold.INCLUDE = True
METHODS = [x for x in METHODS if INCLUDE_HELD or x[0] not in HELD]
_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
COLORS = {m: _cycle[j % len(_cycle)] for j, (m, _) in enumerate(METHODS)}
LABELS = dict(METHODS)

## Loading

`load(axis)` returns every cell of the axis, the anchor's included, and the H3
gate status of every (variant, level). A cell is drawn only if `R.cell_ready`
passes: complete and config-audited. Everything else is listed with its reason.

In [ ]:
def load(axis, protocol="standard"):
    """protocol "candidate": the same ladders WITHOUT terminals, each reward at its
    no-terminals index; the full-data point and the task-reward line are then each
    reward's no-terminals cell at that index (phase2_sweeps.noterm_spec, §14.5)."""
    if protocol == "candidate":
        cells, notes = X.collect_exp(api, P, GRID_WINNERS, axis, protocol="candidate")
        anchor, anotes = R.collect(
            api, P, GRID_WINNERS, sp=P.noterm_spec(P.spec()), program=P.DEFAULT_PROGRAM
        )
    else:
        cells, notes = X.collect_exp(api, P, WINNERS, axis)
        anchor, anotes = R.collect(api, P, WINNERS)
    for (d, m), c in anchor.items():  # the full-data level of every drawn series
        if m in LABELS and m != "task_reward":
            cells[(X._short(d), m, X.FULL)] = c
    oracle = {
        X._short(d): c for (d, m), c in anchor.items() if m == "task_reward"
    }  # the stage-4 winning index (§4.3.157)
    gate = X.gate_status(axis)
    for n in notes + anotes:
        print("note:", n)
    return cells, oracle, gate


def stats(cell):
    x = SCALE * np.array([cell["scores"][s] for s in X.SEEDS])
    return float(x.mean()), float(x.std(ddof=1))


def table(cells, oracle, gate, axis):
    """Mean ± sd per (variant, model) and level; the reason where a cell is not drawn."""
    pos = x_positions(axis, X.VARIANTS[0])  # the figure's left-to-right order
    levels = tuple(sorted(pos, key=lambda lv: pos[lv][0]))
    rows = []
    for v in X.VARIANTS:
        for m, label in METHODS:
            row = {"variant": v, "model": label}
            for lv in levels:
                c = oracle.get(v) if m == "task_reward" else cells.get((v, m, lv))
                if m == "task_reward" and lv != X.FULL:
                    row[lv] = "(reference)"
                    continue
                why = R.cell_ready(c)
                g = gate.get((v, lv), "")
                if why:
                    row[lv] = why
                elif m in BNN and g.startswith("INCOMPLETE"):
                    row[lv] = "H3 gate unread"
                else:
                    mu, sd = stats(c)
                    flag = " (H3: conditional)" if m in BNN and g != "PASS" else ""
                    row[lv] = f"{mu:.1f} ± {sd:.1f}{flag}"
            rows.append(row)
    return pd.DataFrame(rows).set_index(["variant", "model"])

## The figure

`plot_axis` draws a 2 × 2 grid, one panel per variant. Series are offset slightly
along x so that error bars do not overlap; the tick marks the level. A missing
cell breaks its series' line rather than bridging it.

In [ ]:
def x_positions(axis, variant):
    """{level: (x, tick label)}, full data first, most degraded last."""
    if axis == "sparsity":
        levels = (X.FULL,) + X.AXES[axis]["levels"][::-1]  # full, 128, 64, 32, 16
        lab = lambda lv: (
            f"full\n({X.TRAIN_POOL[variant]})"
            if lv == X.FULL
            else str(X.SPARSE_PAIRS[lv])
        )
        return {lv: (float(i), lab(lv)) for i, lv in enumerate(levels)}
    levels = (X.FULL,) + X.AXES[axis]["levels"]  # 0, 0.1, ..., 0.45
    return {
        lv: (X._x_noise(variant, lv), "0\n(full)" if lv == X.FULL else f"{X.FLIP[lv]:g}")
        for lv in levels
    }


def plot_axis(cells, oracle, gate, axis, protocol="standard", figsize=(13, 9)):
    fig, axes = plt.subplots(2, 2, figsize=figsize, sharey=False)
    series = [m for m, _ in METHODS if m != "task_reward"]
    for ax, v in zip(axes.ravel(), X.VARIANTS):
        pos = x_positions(axis, v)
        order = sorted(pos, key=lambda lv: pos[lv][0])
        span = pos[order[-1]][0] - pos[order[0]][0]
        xs_sorted = [pos[lv][0] for lv in order]
        gap = min(b - a for a, b in zip(xs_sorted, xs_sorted[1:]))
        step = 0.5 * gap / len(series)  # dodge: all series within half the closest gap
        drawn = False
        for j, m in enumerate(series):
            dx = (j - (len(series) - 1) / 2) * step
            xs, ys, es, hollow = [], [], [], []
            for lv in order:
                c = cells.get((v, m, lv))
                g = gate.get((v, lv), "")
                ok = R.cell_ready(c) is None and not (
                    m in BNN and g.startswith("INCOMPLETE")
                )
                xs.append(pos[lv][0] + dx)
                if ok:
                    mu, sd = stats(c)
                    ys.append(mu)
                    es.append(sd)
                    hollow.append(m in BNN and g != "PASS")
                else:  # NaN breaks the line
                    ys.append(np.nan)
                    es.append(np.nan)
                    hollow.append(False)
            if all(np.isnan(ys)):
                continue
            drawn = True
            col = COLORS[m]
            ax.plot(xs, ys, "-", color=col, lw=1.4, alpha=0.9, label=LABELS[m])
            ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor=col, capsize=3, lw=1)
            for x, y, h in zip(xs, ys, hollow):
                if not np.isnan(y):
                    ax.plot(
                        x,
                        y,
                        "o",
                        ms=6,
                        color=col,
                        markerfacecolor="white" if h else col,
                        markeredgewidth=1.5,
                    )
        o = oracle.get(v)
        if R.cell_ready(o) is None:
            mu, sd = stats(o)
            ax.axhline(
                mu,
                ls="--",
                color=COLORS["task_reward"],
                lw=1.2,
                label=LABELS["task_reward"],
            )
            ax.axhspan(mu - sd, mu + sd, color=COLORS["task_reward"], alpha=0.08)
            drawn = True
        pad = 0.08 * span + abs(step) * len(series) / 2  # room for the dodged points
        ax.set_xlim(pos[order[0]][0] - pad, pos[order[-1]][0] + pad)
        ax.set_xticks([pos[lv][0] for lv in order])
        ax.set_xticklabels([pos[lv][1] for lv in order])
        ax.set_xlabel("preference pairs" if axis == "sparsity" else "flip fraction p")
        ax.set_ylabel("score (last-10 mean × 100)")
        ax.set_title(f"antmaze-{v}")
        ax.grid(alpha=0.3)
        if not drawn:
            ax.text(
                0.5,
                0.5,
                "no complete cells yet",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
            )
    handles, labels = {}, {}
    for ax in axes.ravel():
        for h, lab in zip(*ax.get_legend_handles_labels()):
            handles.setdefault(lab, h)
    order = [LABELS[m] for m, _ in METHODS if LABELS[m] in handles]
    fig.legend(
        [handles[l] for l in order],
        order,
        loc="lower center",
        ncol=len(order),
        frameon=False,
    )
    title = X.AXES[axis]["title"].replace(" -- ", ": ")
    if protocol == "candidate":
        title += ", WITHOUT terminals (each at its no-terminals index)"
    fig.suptitle(
        f"{title} (mean ± sd over seeds 1-10; hollow = H3-conditional; "
        "dashed = task reward at full data)"
    )
    fig.tight_layout(rect=(0, 0.05, 1, 0.96))
    plt.show()

# Experiment 1: data sparsity

In [ ]:
sp_cells, sp_oracle, sp_gate = load("sparsity")
table(sp_cells, sp_oracle, sp_gate, "sparsity")

In [ ]:
plot_axis(sp_cells, sp_oracle, sp_gate, "sparsity")

# Experiment 2: label noise (primary arm)

Training and test labels flipped at rate p; validation clean (§11). The
sensitivity arm (clean selection split, medium_play, MR and PT only) is not drawn
here. It is descriptive and reported beside the primary result.

In [ ]:
nz_cells, nz_oracle, nz_gate = load("noise")
table(nz_cells, nz_oracle, nz_gate, "noise")

In [ ]:
plot_axis(nz_cells, nz_oracle, nz_gate, "noise")

# Without terminals (§14.5's candidate arm)

The same figures for the **same reward models**, evaluated without the dataset's
terminals (`drop_terminals`). Every reward runs at its own **no-terminals index**,
the winner of its seed-0 no-terminals grid (`results/control_grid_winners.json`),
frozen (`gp_reward-priors/HANDOFF_HP_SELECTION.md` §14.5). The full-data point of
each series, and the dashed task-reward line, are that reward's no-terminals cell at
its own index.

A ladder cell is the experiment's sweep file with `drop_terminals` added and the
index moved to the no-terminals one. It is found by an exact parameter match
(`exp_readout.collect_exp(..., protocol="candidate")`) and drawn under the same
gates as above. **The no-terminals ladders have not been run yet** (§14.4: they come
last), so until then each panel shows only the full-data points and the reference
line.

Descriptive. §14.5's tests (a)-(c) are read by their own readout, not from these
figures.

## Experiment 1 without terminals: data sparsity

In [ ]:
spn_cells, spn_oracle, spn_gate = load("sparsity", "candidate")
table(spn_cells, spn_oracle, spn_gate, "sparsity")

In [ ]:
plot_axis(spn_cells, spn_oracle, spn_gate, "sparsity", "candidate")

## Experiment 2 without terminals: label noise (primary arm)

In [ ]:
nzn_cells, nzn_oracle, nzn_gate = load("noise", "candidate")
table(nzn_cells, nzn_oracle, nzn_gate, "noise")

In [ ]:
plot_axis(nzn_cells, nzn_oracle, nzn_gate, "noise", "candidate")

# Reward-model accuracy on the clean validation split

The same layout for the **reward models themselves**: one panel per variant, one series
per reward model, the x-axis from full data (left) to the most degraded (right). The
y-axis is **per-pair preference accuracy on each seed's clean, tie-free validation split**
(× 100), the quantity A of RQ1's manipulation check (handoff §14.5). Each point is the
mean over seeds 1-10, with the seed-to-seed sd as the error bar.

- **Source:** `exp/rq1_accuracy.json`, measured on the box by `results/rq1_accuracy.py`.
  Each best model is re-scored with its own trainer's evaluation on all pairs at once.
  The ensemble mean's accuracy is that of its mean reward over MR's snapshots from epoch
  100. The file is gitignored and copied over by hand.
- **Gate:** a point is drawn only when all ten seeds are measured and every MR, PT and NMR
  record passes `rq1_readout`'s cross-check against the training run's logged
  `eval_acc_reloaded`. The table gives the reason for every point not drawn.
- **The validation split never changes with the level** (§13.6, §11.3), so every point of
  a panel is scored on the same pairs.
- **Dashed line:** the path-length floor (§13.13). It scores the same pairs by which
  segment's torso travelled further, with no learning; the band is its seed-to-seed sd.

**Descriptive, not a test.** The registered reading is `python results/rq1_readout.py`
(the manipulation check: the degradation slope of A > 0).

In [ ]:
import rq1_readout as Q1  # noqa: E402  A and its cross-check

sys.path.insert(0, os.path.abspath(os.path.join("..", "gp_reward-priors", "scripts_bnn")))
import heuristic_floor as HF  # noqa: E402  the path-length floor (§13.13)

ACC_PATH = os.path.abspath(os.path.join("..", "exp", "rq1_accuracy.json"))
ACC_SERIES = [m for m, _ in METHODS if m != "task_reward"]  # the reward models, in METHODS order


def load_acc():
    """{(FAM, variant, level, seed): accuracy, or the reason it is not read}."""
    acc, notes = Q1.load_accuracy(ACC_PATH, Q1.logged_accuracies() if os.path.exists(ACC_PATH) else None)
    for n in notes:
        print("note:", n)
    return acc


def load_floor():
    """{variant: [ten per-seed floor accuracies]} on the same validation pairs."""
    return {v: [float(HF.floor_correct(HF.val_file(v.replace("-", "_"), s)).mean()) for s in X.SEEDS]
            for v in X.VARIANTS}


def acc_point(acc, m, v, lv):
    """(mean, sd) × 100 over the ten seeds, or (None, reason)."""
    vals = [acc.get((m, v, lv, s)) for s in X.SEEDS]
    bad = [x for x in vals if not isinstance(x, float)]
    if bad:
        return None, f"{10 - len(bad)}/10 measured" + (f" ({bad[0]})" if isinstance(bad[0], str) else "")
    x = SCALE * np.array(vals)
    return (float(x.mean()), float(x.std(ddof=1))), None


def table_acc(acc, floor, axis):
    """Mean ± sd per (variant, model) and level; the reason where a point is not drawn."""
    pos = x_positions(axis, X.VARIANTS[0])
    levels = tuple(sorted(pos, key=lambda lv: pos[lv][0]))
    rows = []
    for v in X.VARIANTS:
        f = SCALE * np.array(floor[v])
        rows.append({"variant": v, "model": "path-length floor",
                     **{lv: f"{f.mean():.1f} ± {f.std(ddof=1):.1f}" for lv in levels}})
        for m in ACC_SERIES:
            row = {"variant": v, "model": LABELS[m]}
            for lv in levels:
                p, why = acc_point(acc, m, v, lv)
                row[lv] = why if p is None else f"{p[0]:.1f} ± {p[1]:.1f}"
            rows.append(row)
    return pd.DataFrame(rows).set_index(["variant", "model"])


def plot_acc(acc, floor, axis, figsize=(13, 9)):
    fig, axes = plt.subplots(2, 2, figsize=figsize, sharey=False)
    for ax, v in zip(axes.ravel(), X.VARIANTS):
        pos = x_positions(axis, v)
        order = sorted(pos, key=lambda lv: pos[lv][0])
        span = pos[order[-1]][0] - pos[order[0]][0]
        xs_sorted = [pos[lv][0] for lv in order]
        gap = min(b - a for a, b in zip(xs_sorted, xs_sorted[1:]))
        step = 0.5 * gap / len(ACC_SERIES)  # dodge, as plot_axis
        drawn = False
        for j, m in enumerate(ACC_SERIES):
            dx = (j - (len(ACC_SERIES) - 1) / 2) * step
            xs, ys, es = [], [], []
            for lv in order:
                p, _ = acc_point(acc, m, v, lv)
                xs.append(pos[lv][0] + dx)
                ys.append(np.nan if p is None else p[0])  # NaN breaks the line
                es.append(np.nan if p is None else p[1])
            if all(np.isnan(ys)):
                continue
            drawn = True
            col = COLORS[m]
            ax.plot(xs, ys, "-", color=col, lw=1.4, alpha=0.9, label=LABELS[m])
            ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor=col, capsize=3, lw=1)
            ax.plot(xs, ys, "o", ms=6, color=col)
        f = SCALE * np.array(floor[v])
        ax.axhline(f.mean(), ls="--", color="0.35", lw=1.2, label="path-length floor")
        ax.axhspan(f.mean() - f.std(ddof=1), f.mean() + f.std(ddof=1), color="0.35", alpha=0.08)
        pad = 0.08 * span + abs(step) * len(ACC_SERIES) / 2
        ax.set_xlim(pos[order[0]][0] - pad, pos[order[-1]][0] + pad)
        ax.set_xticks([pos[lv][0] for lv in order])
        ax.set_xticklabels([pos[lv][1] for lv in order])
        ax.set_xlabel("preference pairs" if axis == "sparsity" else "flip fraction p")
        ax.set_ylabel("validation accuracy (per pair, × 100)")
        ax.set_title(f"antmaze-{v}")
        ax.grid(alpha=0.3)
        if not drawn:
            ax.text(0.5, 0.5, "no complete points yet", ha="center", va="center",
                    transform=ax.transAxes, color="gray")
    handles = {}
    for ax in axes.ravel():
        for h, lab in zip(*ax.get_legend_handles_labels()):
            handles.setdefault(lab, h)
    order = [LABELS[m] for m in ACC_SERIES if LABELS[m] in handles] + ["path-length floor"]
    fig.legend([handles[l] for l in order], order, loc="lower center", ncol=len(order), frameon=False)
    title = X.AXES[axis]["title"].split(" -- ")[1] if " -- " in X.AXES[axis]["title"] else axis
    fig.suptitle(f"Reward-model accuracy, {title}: per pair on the clean validation split "
                 "(mean ± sd over seeds 1-10; dashed = path-length floor)")
    fig.tight_layout(rect=(0, 0.05, 1, 0.96))
    plt.show()


acc = load_acc()
floor = load_floor()

## Accuracy, data sparsity

In [ ]:
table_acc(acc, floor, "sparsity")

In [ ]:
plot_acc(acc, floor, "sparsity")

## Accuracy, label noise (primary arm)

Training and test labels flipped at rate p; the validation split, on which accuracy is
measured here, is clean (§11). The test split chose each checkpoint, so in this arm the
checkpoint was chosen on noisy labels.

In [ ]:
table_acc(acc, floor, "noise")

In [ ]:
plot_acc(acc, floor, "noise")